# Dataset D pancreas validation: acquire and validate public data

This notebook uses the official public CellRank pancreas dataset only. It downloads or loads the CellRank `raw`, `preprocessed`, and `preprocessed-kernel` dataset kinds, verifies required layers and annotations, records file checksums, and writes validation artifacts.

The `preprocessed-kernel` object is recorded only as a VelocityKernel-derived reference/comparator. It is not independent of scVelo RNA velocity.

## Reader guide

- **Purpose:** Dataset D pancreas validation: acquire and validate public data in the frozen revision workflow.
- **Inference scope:** Descriptive public developmental-dynamics validation. CellRank provides velocity-derived context, not independent biological confirmation.
- **Inputs:** The official public pancreas input or the checksum-validated output of the preceding numbered stage.
- **Implementation:** `scripts/pancreas_validation_common.py`.
- **Outputs:** Ignored `results/public_validation/pancreas_dataset_d/` artifacts.
- **Frozen findings:** Retain the frozen descriptive representation–dynamics findings and negative controls; do not claim causal trajectories or universal biological preservation.
- **Limitations:** The workflow is descriptive, depends on supplied dynamics and representations, and does not provide independent biological replication.

This source notebook is intentionally a thin, output-free entry point. The testable implementation is maintained in scripts/pancreas_validation_common.py. Executed review copies and generated artifacts are written under the ignored results directory.


In [ ]:
from pathlib import Path
import os
import sys

ROOT = Path.cwd().resolve()
sys.path.insert(0, str(ROOT / "scripts"))
from pancreas_validation_common import (
    configured_paths,
    ensure_output_tree,
    ensure_runtime_env,
    load_config,
    rel_display,
    sha256_file,
    version_record,
    write_alt_text,
    write_dataframe,
    write_json,
    write_metadata,
)

CONFIG = load_config(ROOT)
PATHS = configured_paths(CONFIG, ROOT)
DATA_DIR = PATHS["data_dir"]
OUTPUT_DIR = PATHS["output_dir"]
DATA_DIR.mkdir(parents=True, exist_ok=True)
ensure_runtime_env(OUTPUT_DIR)
ensure_output_tree(OUTPUT_DIR)

In [ ]:

import json
from datetime import datetime, timezone

import cellrank as cr
import h5py
import pandas as pd
import requests

base_path = DATA_DIR / CONFIG["dataset"]["base_filename"]
kind_rows = []
cluster_rows = []
expected_shapes = CONFIG["dataset"]["expected_shapes"]
required_layers = set(CONFIG["dataset"]["required_layers"])
required_obs = set(CONFIG["dataset"]["required_obs"])
required_obsm = set(CONFIG["dataset"]["required_obsm"])

def dataset_path_for(kind):
    return base_path.with_name(f"{base_path.stem}_{kind}{base_path.suffix}")

def is_valid_h5ad(path):
    if not path.exists() or path.stat().st_size == 0:
        return False
    try:
        with h5py.File(path, "r"):
            return True
    except OSError:
        return False

def invalid_h5ad_cache_removed(path):
    if path.exists() and not is_valid_h5ad(path):
        path.unlink()
        return True
    return False

def download_official_cellrank_file(kind, path):
    url = CONFIG["dataset"]["official_file_urls"][kind]
    tmp_path = path.with_suffix(path.suffix + ".download")
    if tmp_path.exists():
        tmp_path.unlink()
    with requests.get(url, stream=True, timeout=(30, 600)) as response:
        response.raise_for_status()
        with tmp_path.open("wb") as handle:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    handle.write(chunk)
    if not is_valid_h5ad(tmp_path):
        size = tmp_path.stat().st_size if tmp_path.exists() else 0
        raise OSError(f"Downloaded public CellRank pancreas file for {kind} is not a valid H5AD; size={size}")
    tmp_path.replace(path)
    return url

cache_rows = []
for kind in CONFIG["dataset"]["kind_order"]:
    dataset_path = dataset_path_for(kind)
    removed_invalid_cache = invalid_h5ad_cache_removed(dataset_path)
    downloaded_url = None
    if not is_valid_h5ad(dataset_path):
        downloaded_url = download_official_cellrank_file(kind, dataset_path)
    adata = cr.datasets.pancreas(path=base_path, kind=kind)
    missing_layers = sorted(required_layers.difference(adata.layers.keys()))
    missing_obs = sorted(required_obs.difference(adata.obs.columns))
    missing_obsm = sorted(required_obsm.difference(adata.obsm.keys()))
    expected_shape = tuple(expected_shapes[kind])
    shape_ok = tuple(adata.shape) == expected_shape
    cell_count_ok = int(adata.n_obs) == int(CONFIG["dataset"]["expected_cell_count"])
    checksum = sha256_file(dataset_path) if dataset_path.exists() else None
    status = "passed" if shape_ok and cell_count_ok and not missing_layers and not missing_obs and not missing_obsm and checksum else "failed"
    kind_rows.append({
        "kind": kind,
        "path": rel_display(dataset_path, ROOT),
        "n_obs": int(adata.n_obs),
        "n_vars": int(adata.n_vars),
        "expected_n_obs": int(expected_shape[0]),
        "expected_n_vars": int(expected_shape[1]),
        "shape_ok": bool(shape_ok),
        "cell_count_ok": bool(cell_count_ok),
        "missing_layers": ";".join(missing_layers),
        "missing_obs": ";".join(missing_obs),
        "missing_obsm": ";".join(missing_obsm),
        "sha256": checksum,
        "checksum_status": "recorded_and_used_for_downstream_verification" if checksum else "missing_file",
        "removed_invalid_cache_before_download": bool(removed_invalid_cache),
        "downloaded_url": downloaded_url,
        "validation_status": status,
    })
    if CONFIG["cluster_key"] in adata.obs:
        counts = adata.obs[CONFIG["cluster_key"]].astype(str).value_counts().sort_index()
        for cluster, count in counts.items():
            cluster_rows.append({"kind": kind, "cluster": cluster, "n_cells": int(count)})

validation = pd.DataFrame(kind_rows)
cluster_counts = pd.DataFrame(cluster_rows)
checksums = validation[["kind", "path", "sha256", "checksum_status"]].copy()
write_dataframe(OUTPUT_DIR, "00_pancreas_dataset_validation", validation)
write_dataframe(OUTPUT_DIR, "00_pancreas_cluster_counts", cluster_counts)
write_dataframe(OUTPUT_DIR, "00_pancreas_dataset_checksums", checksums)
write_alt_text(
    OUTPUT_DIR,
    "00_pancreas_dataset_validation",
    "Public CellRank pancreas raw, preprocessed, and preprocessed-kernel files were validated for expected shapes, spliced and unspliced layers, cluster annotations, UMAP coordinates, and recorded SHA-256 checksums. The preprocessed-kernel file is a VelocityKernel-derived comparator, not independent evidence."
)
write_metadata(OUTPUT_DIR, "00_acquire_and_validate", CONFIG, {
    "dataset_files": validation.to_dict(orient="records"),
    "cellrank_dataset_function": CONFIG["dataset"]["cellrank_dataset_function"],
    "cellrank_documentation": CONFIG["dataset"]["official_documentation"],
    "official_file_urls": CONFIG["dataset"]["official_file_urls"],
})
version_record(OUTPUT_DIR, "00_acquire_and_validate", CONFIG, {"dataset_checksums": checksums.to_dict(orient="records")})
if not validation["validation_status"].eq("passed").all():
    raise AssertionError("One or more public pancreas dataset validation checks failed")
validation
